In [0]:
%sql
WITH raw_data AS (
  SELECT DISTINCT
    ev3.universe,
    UPPER(TRIM(ev3.table)) AS BO_TABLE,
    CASE
      WHEN UPPER(ev3.SOURCE) LIKE 'ALIAS%' AND ev3.base_table IS NOT NULL AND ev3.base_table != ''
        THEN UPPER(TRIM(ev3.base_table))
      ELSE UPPER(TRIM(ev3.table))
    END AS source_table,
    UPPER(TRIM(ev3.schema)) AS SCHEMA,
    CASE
      WHEN UPPER(ev3.SOURCE) LIKE 'ALIAS%'
        THEN UPPER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(ev3.SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE UPPER(TRIM(ev3.SOURCE))
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction as ev3

  UNION ALL

  SELECT  DISTINCT
    ev4.universe,
    UPPER(TRIM(ev4.table)) AS BO_TABLE,
    CASE
      WHEN UPPER(ev4.SOURCE) LIKE 'ALIAS%' AND ev4.base_table IS NOT NULL AND ev4.base_table != ''
        THEN UPPER(TRIM(ev4.base_table))
      ELSE UPPER(TRIM(ev4.table))
    END AS source_table,
    UPPER(TRIM(ev4.schema)) AS SCHEMA,
    CASE
      WHEN UPPER(ev4.SOURCE) LIKE 'ALIAS%'
        THEN UPPER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(ev4.SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE UPPER(TRIM(ev4.SOURCE))
    END AS SOURCE

  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining as ev4
), 
ranked_source AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY universe, BO_TABLE, SOURCE_TABLE, SCHEMA
      ORDER BY
        CASE 
          WHEN SOURCE = 'SYMPHONY' THEN 1
          WHEN SOURCE = 'ORACLE FINANCE' THEN 2
          WHEN SOURCE = 'ORACLE FINANCE TABLE' THEN 3
          WHEN SOURCE = 'ORACLE FINANCE VIEW' THEN 4
          WHEN SOURCE = 'ORACLEFINANCE' THEN 5
          WHEN SOURCE = 'ORACLE FINANCE MATERIALIZED VIEW' THEN 6
          WHEN SOURCE = 'ORACLE FINACNE' THEN 7
          WHEN SOURCE = 'DW (FACT)' THEN 8
          WHEN SOURCE = 'DW' THEN 9
          WHEN SOURCE = 'DW (DIMENSION)' THEN 10
          WHEN SOURCE = 'DW (VIEW)' THEN 11
          WHEN SOURCE = 'INFORMATICA' THEN 12
          WHEN SOURCE = 'SYSTEM TABLE' THEN 13
          WHEN SOURCE = 'DERIVED' THEN 14
          WHEN SOURCE IN ('NA', 'NOT AVAILABLE') THEN 15
          ELSE 99
        END
    ) AS rn1
  FROM raw_data
),
deduped_source AS (
  SELECT universe, BO_TABLE, SOURCE_TABLE, SCHEMA, SOURCE
  FROM ranked_source
  WHERE rn1 = 1
),
-- Level 2: Deduplicate by (BO_TABLE, SOURCE_TABLE) — keep highest priority SCHEMA
ranked_schema AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY BO_TABLE, SOURCE_TABLE
      ORDER BY
        CASE 
          WHEN SCHEMA = 'ORADMART1' THEN 1
          WHEN SCHEMA = 'ORABUP0' THEN 2
          WHEN SCHEMA = 'AR' THEN 3
          WHEN SCHEMA = 'GL' THEN 4
          WHEN SCHEMA = 'APPS' THEN 5
          WHEN SCHEMA = 'AP' THEN 6
          WHEN SCHEMA = 'CUST' THEN 7
          WHEN SCHEMA = 'FA' THEN 8
          WHEN SCHEMA = 'APPLSYS' THEN 9
          WHEN SCHEMA = 'PO' THEN 10
          WHEN SCHEMA = 'HR' THEN 11
          WHEN SCHEMA = 'XLA' THEN 12
          WHEN SCHEMA = 'JTF' THEN 13
          WHEN SCHEMA = 'INV' THEN 14
          WHEN SCHEMA = 'ZX' THEN 15
          WHEN SCHEMA = 'OKC' THEN 16
          WHEN SCHEMA = 'ICX' THEN 17
          WHEN SCHEMA = 'SYS' THEN 18
          WHEN SCHEMA = 'ORABOFP' THEN 19
          WHEN SCHEMA = 'GBRELS1' THEN 20
          WHEN SCHEMA IN ('NOT AVAILABLE', 'NOT AVILABLE') THEN 21
          ELSE 99
        END
    ) AS rn2
  FROM deduped_source
),
Universe_list as (
  SELECT universe, BO_TABLE, SOURCE_TABLE, SCHEMA, SOURCE
  FROM ranked_schema
  WHERE rn2 = 1
  ORDER BY universe, BO_TABLE, SOURCE_TABLE ASC
)
select 
  upper(trim(Universe_list.universe)) as UNIVERSE,
  upper(trim(Universe_list.BO_TABLE)) as BO_SQL_TABLE, 
  upper(trim(Universe_list.SOURCE_TABLE)) as MI_Table, 
  upper(trim(Universe_list.SCHEMA)) as MI_SCHEMA, 
  upper(trim(Universe_list.SOURCE)) as MI_SOURCE, 
  upper(trim(d1.SOURCE_TABLE)) as Src_table, 
  upper(trim(d1.SOURCE_SCHEMA)) as Src_schema 
from Universe_list 
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.MI_Dictionary_SAP_BO as D1
on upper(trim(Universe_list.SOURCE_TABLE)) = upper(trim(D1.TARGET_TABLE)) 
   and upper(trim(Universe_list.SCHEMA)) = upper(trim(D1.TARGET_SCHEMA))
order by upper(trim(Universe_list.BO_TABLE)) asc, upper(trim(Universe_list.SOURCE_TABLE)) asc


In [0]:
from pyspark.sql import functions as F

# Use the SQL result from cell 1
mi_tables = _sqldf.select("MI_Table", "MI_SCHEMA", "MI_SOURCE", "UNIVERSE", "BO_SQL_TABLE", "Src_table", "Src_schema").distinct()

# Apply refined categorization (18 categories, 95%+ coverage)
categorized_final = mi_tables.withColumn("SUGGESTED_CATEGORY",
    # 1. Policy / Underwriting
    F.when(F.col("MI_Table").rlike("(?i)(^TBPO_|POLICY|POLIC|COVER|PREMIUM|LIMIT|^TBPP_|ENDORSEMENT|DISCRETIONARY|SPECIAL_COND|SPECIAL_PROD|EXPOSURE)"), "Policy / Underwriting")
    # 2. Claims / Collections / Recovery
    .when(F.col("MI_Table").rlike("(?i)(^TBCL_|CLAIM|RECOVERY|OVERDUE|DEBT|COLLECTION|RECOURSE|INDEMNIT)"), "Claims / Collections")
    # 3. Finance / Accounting
    .when(F.col("MI_Table").rlike("(?i)(^GL_|^AP_|^AR_|INVOICE|PAYMENT|RECEIPT|LEDGER|BALANCE|JOURNAL|AGING|PRUDENCE|REINSUR|^TBPA_|ACCOUNTING|CASH_ACCOUNT|AGED_|^RA_CUST|RECEIVABLE|DSO_|CEI|WEIGHTED)"), "Finance / Accounting")
    # 4. Organisation / Reference Data
    .when(F.col("MI_Table").rlike("(?i)(^TBOR_|COUNTRY|COUNTRIES|CURRENCY|ORGANISATION|INDIVIDUAL|SECTOR|ADDRESS|^TPE_|^DD_NCM|^DD_COUNTRY|^DD_CURR|^HZ_|^CG_REF|^TBEN_)"), "Organisation / Reference Data")
    # 5. Sales / Commercial
    .when(F.col("MI_Table").rlike("(?i)(^TBSA_|SALES|COMMERCIAL|PROSPECT|PIPELINE|OPPORTUNITY|MARKET|COMPETITOR|^TBMK_)"), "Sales / Commercial")
    # 6. Buyer / Debtor Management
    .when(F.col("MI_Table").rlike("(?i)(^TBBU_|BUYER|DEBTOR|CREDIT_LIMIT|GRADE|SUBLIMIT)"), "Buyer / Debtor")
    # 7. Premium Management
    .when(F.col("MI_Table").rlike("(?i)(^TBPM_|PREMIUM_CALC|TURNOVER|DECLARATION|BILLING|PRICING)"), "Premium Management")
    # 8. Workflow / User Management
    .when(F.col("MI_Table").rlike("(?i)(^TBWM_|WORKGRP|STEERING|USER|SECURITY|AGENT|DEPT|SECTION|LDAP|^TBSE_|^TBEA_)"), "Workflow / User Management")
    # 9. Risk / Credit Assessment
    .when(F.col("MI_Table").rlike("(?i)(^TBMI_|CSI_|RISK|RATING|SCORE|ASSESSMENT|INFORMATION_SOURCE|INSIGHT)"), "Risk / Credit Assessment")
    # 10. Customer Service
    .when(F.col("MI_Table").rlike("(?i)(^TBCS_|COMPLAINT|SERVICE_REQUEST|TASK|CONTACT_HISTORY)"), "Customer Service")
    # 11. Bonds / Surety
    .when(F.col("MI_Table").rlike("(?i)(^TBBO_|BOND|SURETY|GUARANTEE|PRINCIPAL)"), "Bonds / Surety")
    # 12. Medium-Term / Special Products
    .when(F.col("MI_Table").rlike("(?i)(^TBMT_|MEDIUM_TERM|SINGLE_RISK|EXCESS_OF_LOSS)"), "Medium-Term / Special Products")
    # 13. WITS / IT Service Management
    .when(F.col("MI_Table").rlike("(?i)(^WITS_|^A_WITS|^A_ITGS|^A_SH_WITS|^BAS_|TRADE_INTEL|WORLD_TRADE|PRBM_PROBLEM)"), "WITS / IT Service Management")
    # 14. Data Warehouse / Dimensions
    .when(F.col("MI_Table").rlike("(?i)(^DD_|^DIM_|^FACT_|^FT_|^FC_|^DW_|^AGG_|^MART_|^RPT_|^VW|^VWPA_|^GROUP_|^MAP_|^REG_)"), "Data Warehouse / Dimensions")
    # 15. ERP / System Config
    .when(F.col("MI_Table").rlike("(?i)(^FND_|^APPS_|^SYS_|SYSTEM|LOOKUP|PARAMETER|CONFIG|^OKC_|^PO_|^FA_|^HR_|^XLA_|^ICX_|^MTL_|^PER_ALL|DERIVED)"), "ERP / System Config")
    # 16. Asset Management / Monitoring
    .when(F.col("MI_Table").rlike("(?i)(^TBAM_|ASSET_MGMT|MONITORING)"), "Asset Management / Monitoring")
    # 17. Communications / Correspondence
    .when(F.col("MI_Table").rlike("(?i)(^TBCM_|COMMUNICAT|CORRESPOND|LETTER|DOCUMENT|TEMPLATE)"), "Communications / Correspondence")
    # 18. Reinsurance / Cabrillo
    .when(F.col("MI_Table").rlike("(?i)(^TBGG_|FACULTATIVE|^TBUP_|^CORE_FAC|^RP_FAC|^SS_FAC|UPR|CABRILLO|^WFR_|RECON)"), "Reinsurance / Cabrillo")
    .otherwise("Other / Unclassified")
)

# Write to Unity Catalog
target_table = "dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_table_category"

categorized_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"✓ Written {categorized_final.count()} rows to {target_table}")
display(spark.table(target_table).groupBy("SUGGESTED_CATEGORY").count().orderBy(F.col("count").desc()))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load data items table and category lookup
data_items_raw = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition")
category = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_table_category")

# Clean universe_name: strip '_old_udt', '.unx', and 'Obsolete' suffixes
data_items = data_items_raw \
    .withColumn("universe_name_clean",
        F.trim(F.regexp_replace(F.col("universe_name"), r"(_old_udt|\.unx|\s*Obsolete)$", ""))
    )

# --- Level 1: Match on universe_name → dominant category per universe ---
universe_category = category.groupBy("UNIVERSE", "SUGGESTED_CATEGORY") \
    .agg(F.count("*").alias("table_count")) \
    .withColumn("rank", F.row_number().over(
        Window.partitionBy("UNIVERSE").orderBy(F.col("table_count").desc())
    )) \
    .filter(F.col("rank") == 1) \
    .select(
        F.col("UNIVERSE").alias("cat_universe"),
        F.col("SUGGESTED_CATEGORY").alias("cat_level1")
    )

# --- Level 1b: Normalized match (strip punctuation) ---
universe_category_norm = universe_category.withColumn("cat_universe_norm",
    F.upper(F.trim(F.regexp_replace(F.regexp_replace(F.col("cat_universe"), r"[,.'()&\[\]{}]", " "), r"\s+", " ")))
).select(F.col("cat_universe_norm"), F.col("cat_level1").alias("cat_level1b"))

# --- Level 1c: Manual mapping for known unmatched universe names ---
manual_universe_mapping = {
    # Data Warehouse / Dimensions
    "DALLAS": "Data Warehouse / Dimensions",
    "DALLAS2": "Data Warehouse / Dimensions",
    "BBT_DWH_BO_UNIVERSE": "Data Warehouse / Dimensions",
    "PVDATAMART ATRADIUS": "Data Warehouse / Dimensions",
    "PVDATAMART ATRADIUS NEW": "Data Warehouse / Dimensions",
    "PVPORT": "Data Warehouse / Dimensions",
    "OID PRODUCTION REPORTING": "Data Warehouse / Dimensions",
    "PRODUCTION REPOS UNIVERSE": "Data Warehouse / Dimensions",
    # Finance / Accounting
    "COMBINED AP AR GL WITH XLA": "Finance / Accounting",
    "COMBINED AP,AR & GL UNIVERSE": "Finance / Accounting",
    "FINANCIAL TRANSACTIONS": "Finance / Accounting",
    "PAYABLES": "Finance / Accounting",
    "COMBINED PAYM'T & RECOVERY": "Finance / Accounting",
    "COMBINED PAYM'T _ RECOVERY": "Finance / Accounting",
    "PAYM'T _ RECOVERY FIN": "Finance / Accounting",
    "PAYMT_RECOVERYFIN_DAILY": "Finance / Accounting",
    "EKV 3.2P": "Finance / Accounting",
    "AMT5": "Finance / Accounting",
    "PVDM_BUDGET": "Finance / Accounting",
    "SCHED-2": "Finance / Accounting",
    "SPANISH TAX": "Finance / Accounting",
    "CUMULATIVE CASES FIN_POL YR": "Finance / Accounting",
    # Policy / Underwriting
    "UNDERWRITING MONTHLY BSB": "Policy / Underwriting",
    "UNDERWRITING MONTHLY BSB ADE1": "Policy / Underwriting",
    "POLICY MONTHLY BSB": "Policy / Underwriting",
    "NPP CORE BSB": "Policy / Underwriting",
    "NPP CORE MONTHLY BSB": "Policy / Underwriting",
    "MODULE POLICIES MONTHLY BSB": "Policy / Underwriting",
    "NEW EXPOSURE SKELETON": "Policy / Underwriting",
    # Buyer / Debtor
    "CREDIT LIMITS PROCESSED MONTHLY": "Buyer / Debtor",
    # Claims / Collections
    "CER CLAIMS": "Claims / Collections",
    "GESTION RECOUVREMENT": "Claims / Collections",
    "OVERDUES CLAIMS MONTHLY BSB": "Claims / Collections",
    # WITS / IT Service Management
    "WITS CONTRACTS AND PURCHASING": "WITS / IT Service Management",
    "QUALITY CENTER": "WITS / IT Service Management",
    # ERP / System Config
    "FHSQL034": "ERP / System Config",
    "FHSQL036": "ERP / System Config",
    "FHSQL041": "ERP / System Config",
    "FHSQL043": "ERP / System Config",
    "FHSQL046": "ERP / System Config",
    "FHSQL047": "ERP / System Config",
    "FHSQL048": "ERP / System Config",
    "FHSQL050": "ERP / System Config",
    "FHSQL050 (2)": "ERP / System Config",
    "FHSQL051": "ERP / System Config",
    "FHSQL052 (2)": "ERP / System Config",
    "SQL_INVENTORY": "ERP / System Config",
    # Workflow / User Management
    "GROUP HR": "Workflow / User Management",
    # Organisation / Reference Data
    "TPE": "Organisation / Reference Data",
    "BELGIUM CUSTOMERS": "Organisation / Reference Data",
    # Customer Service
    "CUSTOMER SERVICE CENTER BY TYPE": "Customer Service",
    # Reinsurance / Cabrillo
    "UPR DAC": "Reinsurance / Cabrillo",
}

manual_mapping_expr = F.lit(None).cast("string")
for name, cat in manual_universe_mapping.items():
    manual_mapping_expr = F.when(
        F.upper(F.trim(F.col("universe_name_clean"))) == name, F.lit(cat)
    ).otherwise(manual_mapping_expr)

# --- Level 2: Match SQL_Tables_str against MI_Table in category ---
# SQL_Tables_str contains values like "APPS.GL_CODE_COMBINATIONS", "TBOR_COUNTRIES"
# Extract the table name (after the dot, or the full string if no dot)
data_items_enriched = data_items \
    .withColumn("sql_table_extracted",
        F.upper(F.trim(
            F.when(F.col("SQL_Tables_str").contains("."),
                F.regexp_extract(F.col("SQL_Tables_str"), r"[^.]+\.([^.,]+)", 1)
            ).otherwise(F.col("SQL_Tables_str"))
        ))
    ) \
    .withColumn("univ_normalized",
        F.upper(F.trim(F.regexp_replace(F.regexp_replace(F.col("universe_name_clean"), r"[,.'()&\[\]{}]", " "), r"\s+", " ")))
    )

# Category per MI_Table (deduplicated)
mi_table_category = category.groupBy("MI_Table", "SUGGESTED_CATEGORY") \
    .agg(F.count("*").alias("cnt")) \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("MI_Table").orderBy(F.col("cnt").desc())
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.col("MI_Table").alias("mi_tbl"),
        F.col("SUGGESTED_CATEGORY").alias("cat_level2")
    )

# Category per BO_SQL_TABLE (for schema-prefixed matches like "APPS.GL_CODE_COMBINATIONS")
# Also try matching the full SQL_Tables_str as BO_SQL_TABLE
bo_table_category = category.groupBy("BO_SQL_TABLE", "SUGGESTED_CATEGORY") \
    .agg(F.count("*").alias("cnt")) \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("BO_SQL_TABLE").orderBy(F.col("cnt").desc())
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.col("BO_SQL_TABLE").alias("bo_tbl"),
        F.col("SUGGESTED_CATEGORY").alias("cat_level3")
    )

# --- Multi-level join ---
data_items_categorized = data_items_enriched \
    .join(universe_category, F.upper(F.trim(F.col("universe_name_clean"))) == F.col("cat_universe"), "left") \
    .join(universe_category_norm, F.col("univ_normalized") == F.col("cat_universe_norm"), "left") \
    .join(mi_table_category, F.col("sql_table_extracted") == F.col("mi_tbl"), "left") \
    .join(bo_table_category, F.upper(F.trim(F.col("SQL_Tables_str"))) == F.col("bo_tbl"), "left") \
    .withColumn("manual_cat", manual_mapping_expr) \
    .withColumn("SUGGESTED_CATEGORY",
        F.coalesce(
            F.col("cat_level1"),    # Level 1: Universe name exact match (after suffix strip)
            F.col("cat_level1b"),   # Level 1b: Normalized universe name match
            F.col("manual_cat"),    # Level 1c: Manual mapping
            F.col("cat_level2"),    # Level 2: SQL table → MI_Table match
            F.col("cat_level3")     # Level 3: Full SQL_Tables_str → BO_SQL_TABLE match
        )
    ) \
    .select(
        data_items_enriched["DataItem_unique_Id"],
        data_items_enriched["universe_id"],
        data_items_enriched["DataItem_id"],
        data_items_enriched["DataItem_name"],
        data_items_enriched["universe_name"],
        data_items_enriched["DataItem_type"],
        data_items_enriched["DataItem_description"],
        data_items_enriched["DataItem_dataType"],
        data_items_enriched["DataItem_path"],
        data_items_enriched["universe_path"],
        data_items_enriched["universe_type"],
        data_items_enriched["sql_definition"],
        data_items_enriched["DataItem_name_definition"],
        data_items_enriched["DataItem_path_definition"],
        data_items_enriched["SQL_Tables_str"],
        F.col("SUGGESTED_CATEGORY")
    )

# Write result
target = "dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition"
data_items_categorized.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)

# Summary
total = data_items_categorized.count()
matched = data_items_categorized.filter(F.col("SUGGESTED_CATEGORY").isNotNull()).count()
print(f"✓ Written {total:,} rows to {target}")
print(f"  Category filled: {matched:,} ({100*matched/total:.1f}%)")
print(f"  Category NULL:   {total-matched:,} ({100*(total-matched)/total:.1f}%)")
print(f"\n=== CATEGORY DISTRIBUTION ===")
display(
    data_items_categorized.groupBy("SUGGESTED_CATEGORY")
    .agg(F.countDistinct("universe_name").alias("universes"), F.count("*").alias("data_items"))
    .orderBy(F.col("data_items").desc())
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load the lineage and category tables
lineage_raw = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")
category = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_table_category")

# Clean DataSource_Name and Universe_Name: strip '_old_udt' suffix and '.unx' extension
lineage = lineage_raw \
    .withColumn("DataSource_Name", F.regexp_replace(F.col("DataSource_Name"), r"(_old_udt|\.unx)$", "")) \
    .withColumn("Universe_Name", F.regexp_replace(F.col("Universe_Name"), r"(_old_udt|\.unx)$", ""))

# --- Level 1: Dominant category per universe name ---
universe_category = category.groupBy("UNIVERSE", "SUGGESTED_CATEGORY") \
    .agg(F.count("*").alias("table_count")) \
    .withColumn("rank", F.row_number().over(
        Window.partitionBy("UNIVERSE").orderBy(F.col("table_count").desc())
    )) \
    .filter(F.col("rank") == 1) \
    .select(
        F.col("UNIVERSE").alias("cat_universe"),
        F.col("SUGGESTED_CATEGORY").alias("cat_level1")
    )

# --- Level 2: Match on DataSource_Name (often contains the real universe) ---
universe_category_ds = universe_category.select(
    F.col("cat_universe").alias("ds_universe"),
    F.col("cat_level1").alias("cat_level2")
)

# --- Level 3: Extract table name from sql_definition and match against category MI_Table ---
# sql_definition patterns: "TABLE_NAME.COLUMN", "SUM(TABLE_NAME.COL)", etc.
# Extract the first TABLE_NAME reference
lineage_with_sql_table = lineage \
    .withColumn("sql_table_extracted",
        F.upper(F.trim(F.regexp_extract(F.col("sql_definition"), r"([A-Za-z_][A-Za-z0-9_]*)\.", 1)))
    )

# Get distinct category per MI_Table (take first match by table count)
mi_table_category = category.groupBy("MI_Table", "SUGGESTED_CATEGORY") \
    .agg(F.count("*").alias("cnt")) \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("MI_Table").orderBy(F.col("cnt").desc())
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.col("MI_Table").alias("mi_tbl"),
        F.col("SUGGESTED_CATEGORY").alias("cat_level3")
    )

# Also match BO_SQL_TABLE for broader coverage
bo_table_category = category.groupBy("BO_SQL_TABLE", "SUGGESTED_CATEGORY") \
    .agg(F.count("*").alias("cnt")) \
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("BO_SQL_TABLE").orderBy(F.col("cnt").desc())
    )) \
    .filter(F.col("rn") == 1) \
    .select(
        F.col("BO_SQL_TABLE").alias("bo_tbl"),
        F.col("SUGGESTED_CATEGORY").alias("cat_level4")
    )

# --- Level 5: Manual mapping for known unmatched names (by DataSource_Name context) ---
# Map based on domain knowledge of SAP BO universe/datasource naming
manual_mapping = {
    # Finance / Accounting
    "COMBINED AP,AR & GL UNIVERSE": "Finance / Accounting",
    "APARGL11": "Finance / Accounting",
    "FINANCIAL TRANSACTIONS": "Finance / Accounting",
    "FTTRANS": "Finance / Accounting",
    "PRUDENCE PROD (ORFP)": "Finance / Accounting",
    "PRUDENCE (ORFP) PROD": "Finance / Accounting",
    "INVOICE TYPES": "Finance / Accounting",
    "PAYM'T _ RECOVERY FIN": "Finance / Accounting",
    "NEW_AP_DATA": "Finance / Accounting",
    "NEW_AR_DATA": "Finance / Accounting",
    "NEW_GL_DATA": "Finance / Accounting",
    "NEW_GL_DATA_EURO": "Finance / Accounting",
    "NEW_AR_DATA_EURO": "Finance / Accounting",
    "CONTANTS - LEDGER": "Finance / Accounting",
    "CONTANTS - LEGAL ENTITY": "Finance / Accounting",
    "CONTANTS - ORGANIZATION": "Finance / Accounting",
    "GL1": "Finance / Accounting",
    "ACCOUNTS": "Finance / Accounting",
    # Policy / Underwriting
    "UNDERWRITING MONTHLY BSB": "Policy / Underwriting",
    "POLICY MONTHLY BSB": "Policy / Underwriting",
    "POLICY  MONTHLY BSB": "Policy / Underwriting",
    # Claims / Collections
    "COMBINED PAYM'T & RECOVERY": "Claims / Collections",
    # Organisation / Reference Data
    "ORGANISATIONS": "Organisation / Reference Data",
    "ORG ID": "Organisation / Reference Data",
    "REF TABLE": "Organisation / Reference Data",
    "COUNTRY": "Organisation / Reference Data",
    # Data Warehouse / Dimensions
    "DWH PRODUCTION 1": "Data Warehouse / Dimensions",
    "PVDATAMART ATRADIUS": "Data Warehouse / Dimensions",
    "DALLAS": "Data Warehouse / Dimensions",
    # Risk / Credit Assessment
    "SYMP_RED_HOT": "Risk / Credit Assessment",
    # ERP / System Config
    "FHSQL046": "ERP / System Config",
    "FHSQL050": "ERP / System Config",
    "FHSQL050 (2)": "ERP / System Config",
}

# Build a mapping column from both Universe_Name and DataSource_Name
manual_mapping_expr_univ = F.lit(None).cast("string")
for name, cat in manual_mapping.items():
    manual_mapping_expr_univ = F.when(
        F.upper(F.trim(F.col("Universe_Name"))) == name, F.lit(cat)
    ).otherwise(manual_mapping_expr_univ)

manual_mapping_expr_ds = F.lit(None).cast("string")
for name, cat in manual_mapping.items():
    manual_mapping_expr_ds = F.when(
        F.upper(F.trim(F.col("DataSource_Name"))) == name, F.lit(cat)
    ).otherwise(manual_mapping_expr_ds)

# --- Level 1b: Normalized join (strip punctuation before matching) ---
# Normalize: remove commas, apostrophes, parentheses, &, collapse spaces
lineage_with_sql_table = lineage_with_sql_table \
    .withColumn("univ_normalized",
        F.upper(F.trim(F.regexp_replace(F.regexp_replace(F.col("Universe_Name"), r"[,.'()&\[\]{}]", " "), r"\s+", " ")))
    ) \
    .withColumn("ds_normalized",
        F.upper(F.trim(F.regexp_replace(F.regexp_replace(F.col("DataSource_Name"), r"[,.'()&\[\]{}]", " "), r"\s+", " ")))
    )

# Normalized category lookup
universe_category_norm = universe_category.withColumn("cat_universe_norm",
    F.upper(F.trim(F.regexp_replace(F.regexp_replace(F.col("cat_universe"), r"[,.'()&\[\]{}]", " "), r"\s+", " ")))
).select(F.col("cat_universe_norm"), F.col("cat_level1").alias("cat_level1b"))

# --- Multi-level join ---
variable_categorized = lineage_with_sql_table \
    .join(universe_category, F.upper(F.trim(F.col("Universe_Name"))) == F.col("cat_universe"), "left") \
    .join(universe_category_ds, F.upper(F.trim(F.col("DataSource_Name"))) == F.col("ds_universe"), "left") \
    .join(universe_category_norm, F.col("univ_normalized") == F.col("cat_universe_norm"), "left") \
    .join(mi_table_category, F.col("sql_table_extracted") == F.col("mi_tbl"), "left") \
    .join(bo_table_category, F.col("sql_table_extracted") == F.col("bo_tbl"), "left") \
    .withColumn("manual_cat_univ", manual_mapping_expr_univ) \
    .withColumn("manual_cat_ds", manual_mapping_expr_ds) \
    .withColumn("UNIVERSE_PRIMARY_CATEGORY",
        F.coalesce(
            F.col("cat_level1"),     # Level 1: Universe_Name exact match
            F.col("cat_level1b"),    # Level 1b: Universe_Name normalized match
            F.col("cat_level2"),     # Level 2: DataSource_Name exact match
            F.col("manual_cat_univ"),# Level 5a: Manual mapping on Universe_Name
            F.col("manual_cat_ds"),  # Level 5b: Manual mapping on DataSource_Name
            F.col("cat_level3"),     # Level 3: sql_definition table → MI_Table
            F.col("cat_level4")      # Level 4: sql_definition table → BO_SQL_TABLE
        )
    ) \
    .select(
        lineage_with_sql_table["Document_Id"],
        lineage_with_sql_table["variable_id"],
        lineage_with_sql_table["variable_name"],
        lineage_with_sql_table["variable_qualification"],
        lineage_with_sql_table["variable_definition"],
        lineage_with_sql_table["extracted_datafield"],
        lineage_with_sql_table["datafield_name"],
        lineage_with_sql_table["DataSource_Name"],
        lineage_with_sql_table["Universe_Name"],
        lineage_with_sql_table["sql_definition"],
        F.col("UNIVERSE_PRIMARY_CATEGORY")
    )

# Summary stats
total_vars = variable_categorized.count()
distinct_docs = variable_categorized.select("Document_Id").distinct().count()
distinct_vars = variable_categorized.select("Document_Id", "variable_id").distinct().count()
matched = variable_categorized.filter(F.col("UNIVERSE_PRIMARY_CATEGORY").isNotNull()).count()

print(f"Total rows: {total_vars}")
print(f"Distinct Documents: {distinct_docs}")
print(f"Distinct Variables (Doc + Var): {distinct_vars}")
print(f"\nMatched (has category): {matched} ({100*matched/total_vars:.1f}%)")
print(f"Unmatched (NULL):       {total_vars - matched} ({100*(total_vars-matched)/total_vars:.1f}%)")
print(f"\n=== CATEGORY DISTRIBUTION ACROSS VARIABLES ===")

display(
    variable_categorized
    .groupBy("UNIVERSE_PRIMARY_CATEGORY")
    .agg(
        F.countDistinct("Document_Id").alias("documents"),
        F.countDistinct("Document_Id", "variable_id").alias("unique_variables"),
        F.count("*").alias("total_rows")
    )
    .orderBy(F.col("unique_variables").desc())
)

In [0]:
from pyspark.sql import functions as F

# Build full output: all 21 original columns + UNIVERSE_PRIMARY_CATEGORY
# Re-select all original columns from lineage_with_sql_table (cleaned DataSource/Universe names)
# and add the category from the multi-level join
original_cols = [
    "Document_Id", "variable_id", "variable_name", "variable_datatype",
    "variable_qualification", "variable_definition", "extracted_datafield",
    "provider_qualifier", "Data_Provider_ID", "datafield_id", "datafield_name",
    "datafield_qualification", "DataSource_Type", "DataSource_Name",
    "datafield_datasourceObjectId", "Universe_ID", "Universe_Name",
    "webi_all_sql_queries", "datafield_datasourcePath", "datafield_description",
    "sql_definition"
]

# Rebuild with all columns from cell 4 logic (reuse variable_categorized join)
full_output = lineage_with_sql_table \
    .join(universe_category, F.upper(F.trim(F.col("Universe_Name"))) == F.col("cat_universe"), "left") \
    .join(universe_category_ds, F.upper(F.trim(F.col("DataSource_Name"))) == F.col("ds_universe"), "left") \
    .join(universe_category_norm, F.col("univ_normalized") == F.col("cat_universe_norm"), "left") \
    .join(mi_table_category, F.col("sql_table_extracted") == F.col("mi_tbl"), "left") \
    .join(bo_table_category, F.col("sql_table_extracted") == F.col("bo_tbl"), "left") \
    .withColumn("manual_cat_univ", manual_mapping_expr_univ) \
    .withColumn("manual_cat_ds", manual_mapping_expr_ds) \
    .withColumn("UNIVERSE_PRIMARY_CATEGORY",
        F.coalesce(
            F.col("cat_level1"),
            F.col("cat_level1b"),
            F.col("cat_level2"),
            F.col("manual_cat_univ"),
            F.col("manual_cat_ds"),
            F.col("cat_level3"),
            F.col("cat_level4")
        )
    ) \
    .select(*[lineage_with_sql_table[c] for c in original_cols] + [F.col("UNIVERSE_PRIMARY_CATEGORY")])

# Write back to the same table (overwrite with new schema)
target = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage"

full_output.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target)

print(f"\u2713 Written {full_output.count()} rows to {target}")
print(f"  Columns: {len(original_cols) + 1} (21 original + UNIVERSE_PRIMARY_CATEGORY)")
print(f"\nVerification:")
result = spark.table(target)
print(f"  Table row count: {result.count()}")
print(f"  Category filled: {result.filter(F.col('UNIVERSE_PRIMARY_CATEGORY').isNotNull()).count()}")
print(f"  Category NULL:   {result.filter(F.col('UNIVERSE_PRIMARY_CATEGORY').isNull()).count()}")

In [0]:
%sql
select count(distinct universe_name)
 from dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition
-- limit 100